# NB 25 — Wyoming Agricultural Baseline Pull (Wave 6, W6-A)

This notebook builds a county-level, source-explicit baseline. It does not substitute estimates when a required source cannot be fetched. All blocked pulls are cached as attempt evidence and registered in `MANUAL_FETCH.md`.

In [1]:
from pathlib import Path
import os, subprocess

ROOT = Path.cwd()
if not (ROOT / 'notebooks').exists():
    ROOT = Path.cwd().parent
NB_PATH = ROOT / 'notebooks/25_wy_ag_baseline_pull.ipynb'
branch = subprocess.check_output(['git', 'branch', '--show-current'], cwd=ROOT, text=True).strip()
candidates = [p for p in (ROOT / 'notebooks').glob('25*.ipynb') if p != NB_PATH]
print(f'pwd: {ROOT}')
print(f'branch: {branch}')
print(f'NB 25 pre-write availability: {not candidates}; conflicting candidates: {candidates}')
assert branch == 'w5-verify', f'expected w5-verify, got {branch}'
assert not candidates, 'NB 25 is not free for this ticket'

pwd: /Users/dylanhartman/projects/Energy Modeling/w5-verify
branch: w5-verify
NB 25 pre-write availability: True; conflicting candidates: []


In [2]:
import csv, json
from datetime import date

RAW = ROOT / 'data/raw'
PROCESSED = ROOT / 'data/processed'
RAW.mkdir(exist_ok=True)
PROCESSED.mkdir(exist_ok=True)
VINTAGE = str(date.today())

fiscal = json.loads((PROCESSED / 'wy_county_fiscal_baseline.json').read_text())
WY_COUNTIES = {geoid: record['county_name'] for geoid, record in fiscal['counties'].items()}
assert len(WY_COUNTIES) == 23

SOURCES = [
    ('census_ag_2022', 'Census of Agriculture 2022 county tables', 'https://www.nass.usda.gov/AgCensus/', 'land use, operations, producers, cattle, forage, irrigated acres'),
    ('nass_quickstats', 'USDA NASS Quick Stats', 'https://quickstats.nass.usda.gov/api/api_GET/', 'Wyoming beef cows state total and county cattle/forage series'),
    ('blm_ras', 'BLM Rangeland Administration System', 'https://reports.blm.gov/reports/ras', 'allotment identifier, county, authorized AUM, acreage'),
    ('usfs_grazing', 'USFS grazing allotments', 'https://data.fs.usda.gov/geodata/edw/datasets.php', 'allotment identifier, county, authorized AUM, acreage'),
    ('wy_water_rights', 'Wyoming State Engineer water rights', 'https://seo.wyo.gov/divisions/water-rights', 'county diversion acre-feet and consumptive-use acre-feet'),
    ('rap', 'Rangeland Analysis Platform', 'https://rangelands.app/', 'county invasive annual grass cover current value and 10-year trend'),
    ('bea_farm_proprietor_income', 'BEA CAINC4 farm proprietors income', 'https://apps.bea.gov/itable/?ReqID=70&step=1', 'county farm proprietor income'),
    ('wy_dor_ag_valuation', 'Wyoming DOR agricultural valuation', 'https://dptax.wyo.gov/annual-reports', 'productive-value coefficients by acre class'),
]

attempts = []
for source_id, title, url, fields in SOURCES:
    attempts.append({'source_id': source_id, 'url': url, 'fields_needed': fields, 'vintage': VINTAGE,
                     'status': 'sandbox_blocked', 'error': 'DNS resolution unavailable in execution sandbox; no source value used.'})
(RAW / f'wy_ag_pull_attempts_{VINTAGE}.json').write_text(json.dumps(attempts, indent=2) + '\n')
print(f'cached blocked-pull evidence: {len(attempts)} source attempts')

cached blocked-pull evidence: 8 source attempts


In [3]:
def datum(source_id, unit, confidence='low', notes='Blocked in sandbox; see MANUAL_FETCH.md.'):
    return {'value': None, 'unit': unit, 'year': None, 'source': source_id, 'confidence': confidence, 'notes': notes}

def county_record(geoid, county_name):
    return {
        'geoid': geoid, 'county_name': county_name, 'state': 'WY',
        'land_by_use': {
            'county_area_acres': datum('census_ag_2022', 'acres'),
            'cropland_acres': datum('census_ag_2022', 'acres'),
            'irrigated_acres': datum('census_ag_2022', 'acres'),
            'pastureland_acres': datum('census_ag_2022', 'acres'),
            'rangeland_acres': datum('census_ag_2022', 'acres'),
            'woodland_acres': datum('census_ag_2022', 'acres'),
            'land_in_farms_acres': datum('census_ag_2022', 'acres'),
            'other_land_acres': {**datum('census_ag_2022', 'acres'), 'is_residual_pool': True,
                'derivation': 'county_area_acres - enumerated land-use classes; never absorbs reconciliation error.'},
            'reconciliation_error_acres': datum('census_ag_2022', 'acres', notes='Must equal zero before a populated pull is accepted.'),
        },
        'cattle_and_forage': {
            'beef_cows_head': datum('nass_quickstats', 'head'),
            'forage_acres': datum('census_ag_2022', 'acres'),
            'stocking_rate_aum_per_acre': datum('nass_quickstats', 'AUM/acre', 'low', 'Documented local stocking rate required; capacity is not inferred.'),
            'aum_capacity': datum('nass_quickstats', 'AUM/year', 'low', 'Derived only as forage acres × documented stocking rate.'),
        },
        'federal_aums': {'blm_authorized_aum': datum('blm_ras', 'AUM/year'), 'usfs_authorized_aum': datum('usfs_grazing', 'AUM/year'), 'federal_aum_share': datum('blm_ras + usfs_grazing', 'fraction')},
        'water': {'diversion_acre_feet': datum('wy_water_rights', 'acre-feet/year'), 'consumptive_use_acre_feet': datum('wy_water_rights', 'acre-feet/year')},
        'rap_invasive_cover': {'current_cover_pct': datum('rap', 'percent'), 'ten_year_trend_pct_points': datum('rap', 'percentage-points/10-years'), 'geography': 'county aggregate only; no sub-county invasive claim'},
        'ag_economics': {'operations_count': datum('census_ag_2022', 'operations'), 'producers_count': datum('census_ag_2022', 'producers'), 'farm_proprietor_income_usd': datum('bea_farm_proprietor_income', 'USD/year')},
        'dor_ag_productive_value_coefficients_usd_per_acre': {'linked_ledger': 'data/processed/wy_fiscal_coefficients.json', 'irrigated': datum('wy_dor_ag_valuation', 'USD/acre'), 'dryland': datum('wy_dor_ag_valuation', 'USD/acre'), 'grazing_land': datum('wy_dor_ag_valuation', 'USD/acre')},
    }

baseline = {
    'schema_version': 'wy_county_ag_baseline_v1', 'vintage': VINTAGE,
    'status': 'blocked_pending_manual_source_acquisition',
    'consumers': ['AG1 Dispatch — Wave 6 agricultural scenario layer'],
    'contract_notes': ['Water is represented only as separate diversion and consumptive-use fields.', 'Ag economics use only Census of Agriculture operations/producers and BEA farm proprietor income (D4).', 'DOR productive-value fields are coefficients and link to the fiscal ledger; assessed values are not duplicated.'],
    'counties': {geoid: county_record(geoid, name) for geoid, name in sorted(WY_COUNTIES.items())},
    'credibility_gate': {
        'status': 'BLOCKED',
        'rows': [
            {'check': 'Fremont and Park lead irrigated acreage', 'result': 'BLOCKED', 'investigation': 'Requires Census 2022 county irrigated acreage.'},
            {'check': 'Campbell and Carbon are rangeland-dominated with large federal AUM shares', 'result': 'BLOCKED', 'investigation': 'Requires Census land use plus BLM/USFS AUMs.'},
            {'check': 'Teton land-in-farms is a small fraction of county area', 'result': 'BLOCKED', 'investigation': 'Requires Census county area and land-in-farms fields.'},
            {'check': 'Statewide beef cows within 10% of NASS state total', 'result': 'BLOCKED', 'investigation': 'Requires NASS county and state beef-cow series.'},
        ],
    },
}
(PROCESSED / 'wy_county_ag_baseline.json').write_text(json.dumps(baseline, indent=2) + '\n')
print('wrote 23 null-safe county records; no estimates substituted')

wrote 23 null-safe county records; no estimates substituted


In [4]:
with (PROCESSED / 'wy_grazing_allotments.csv').open('w', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=['allotment_id', 'agency', 'allotment_name', 'county_geoid', 'county_name', 'authorized_aum', 'authorized_use_acres', 'source_url', 'vintage', 'status', 'manual_fetch_id'], lineterminator='\n')
    writer.writeheader()
with (PROCESSED / 'wy_ag_sources.csv').open('w', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=['source_id', 'title', 'source_url', 'fields_needed', 'vintage', 'fetch_status'], lineterminator='\n')
    writer.writeheader()
    for source_id, title, url, fields in SOURCES:
        writer.writerow({'source_id': source_id, 'title': title, 'source_url': url, 'fields_needed': fields, 'vintage': VINTAGE, 'fetch_status': 'sandbox_blocked'})
print('wrote empty-but-schema-valid allotment detail and 8-source registry')

wrote empty-but-schema-valid allotment detail and 8-source registry


In [5]:
manual = ROOT / 'MANUAL_FETCH.md'
heading = f'## W6-A — Wyoming Agricultural Baseline Pull ({VINTAGE})'
entry = f'''
{heading}

All endpoints below were DNS-blocked in the execution sandbox. No values were invented. Cache downloaded source files under `data/raw/` with the pull date, then replace only the corresponding `null` fields in `data/processed/wy_county_ag_baseline.json`.

'''
for source_id, title, url, fields in SOURCES:
    entry += f'- **{source_id} — {title}:** `{url}`. Fields needed: {fields}.\n'
if heading not in manual.read_text():
    manual.write_text(manual.read_text() + entry)
print('appended 8 exact manual-fetch requirements')

appended 8 exact manual-fetch requirements


In [6]:
# P / 0 / scope-audit / review evidence: executable contract checks.
import subprocess
result = subprocess.run(['python', '-m', 'pytest', 'tests/test_wy_ag_baseline_contract.py', '-q'], cwd=ROOT, text=True, capture_output=True)
print(result.stdout)
assert result.returncode == 0, result.stderr
print('Review gates: D4 absence, water separation, residual handling, low AUM confidence, ledger linkage, consumers, and manifest hashes checked.')

...                                                                      [100%]
3 passed in 0.01s

Review gates: D4 absence, water separation, residual handling, low AUM confidence, ledger linkage, consumers, and manifest hashes checked.


---
## W6-A Resume — Actual Fetch Results (2026-07-17)

The initial Codex run had all 8 endpoints DNS-blocked in the sandboxed subprocess. Claude Code retried from its own network context. Results below.


In [ ]:
# W6-A Resume: summary of actual fetch results and sanity table
# This cell is documentation only; data was populated by Claude Code external to the notebook.
print('W6-A resume fetch results documented above.')


=== NASS COA 2022 BULK PULL — SUCCESS ===
Source: https://www.nass.usda.gov/datasets/qs.census2022.txt.gz (295MB)
Filter: STATE_FIPS_CODE=56, AGG_LEVEL_DESC=COUNTY, SOURCE_DESC=CENSUS
Rows extracted: 9893
Counties populated: 23 / 23

=== ENDPOINT PROBE RESULTS ===
census_ag_2022      : SUCCESS — bulk download path
nass_quickstats     : BLOCKED — api_key_required (HTTP 401)
blm_ras             : BLOCKED — no_programmatic_api (web portal only; gis.blm.gov 404)
usfs_grazing        : BLOCKED — download_available (50MB shapefile; manual download req)
wy_water_rights     : BLOCKED — url_404 (seo.wyo.gov/divisions/water-rights)
rap                 : BLOCKED — no_public_api (rangelands.app/api/ 404; api.rangelands.app DNS fail)
bea_farm_proprietor : BLOCKED — api_key_required (BEA API code 1)
wy_dor_ag_valuation : BLOCKED — dns_not_found (dptax.wyo.gov)

=== SANITY TABLE ===
Check 1 — Fremont/Park lead irrigated: INVESTIGATION REQUIRED
  2022 COA ranking: Carbon(157,301) > Sublette(120,242) > 

---
# C-3 — BLM grazing allotments and AUMs (W7-F3, 2026-11-16)

**Added to NB 25 rather than a new notebook**: NB 25 is the AG0 baseline notebook and owns
`wy_county_ag_baseline.json`, which this section extends to schema `wy_county_ag_baseline_v2`.

Logic lives in `scripts/build_c3_blm_grazing.py` (testable, and `geopandas` overlay code is
awkward to keep in cells); this section runs it and prints every table the ticket asks for.
The earlier cells of this notebook are the AG0 record. Cell 1 asserts branch `w5-verify` and
cannot run on `main`, so the C-3 cells below are self-contained and runnable on their own.

Judgment calls (full text in `c3_blm.judgment_calls` of the baseline file): **JC-1** RAS
"Permitted AUMs" = active AUMs; **JC-2** AUMs apportioned to counties by polygon area share
(low-medium confidence); **JC-3** the public-land proportion scales acres, not AUMs.
**Status: not promoted.** W7-F5 amended the credibility gate (per-county acres vs independent BLM surface acres; statewide AUMs): 22/23 counties pass, Johnson is a recorded exception (DECISIONS.md 2026-11-16). Promotion is deferred to the first session of the next version.

In [1]:
# C-3 cell 1 — pwd / branch
from pathlib import Path
import subprocess, sys
ROOT = Path.cwd()
if not (ROOT / 'notebooks').exists():
    ROOT = Path.cwd().parent
print('pwd:', ROOT)
print('branch:', subprocess.check_output(['git', 'branch', '--show-current'], cwd=ROOT, text=True).strip())
sys.path.insert(0, str(ROOT / 'scripts'))
import json, pandas as pd
pd.set_option('display.width', 220); pd.set_option('display.max_rows', 100)
import build_c3_blm_grazing as c3

pwd: /Users/dylanhartman/projects/Energy Modeling/energy-map
branch: main


In [2]:
# C-3 cell 2 — before-snapshot of the federal-AUM fields (standing protocol: versioned before/after)
snap = json.loads((c3.SNAP / 'before_federal_aum.json').read_text())
print('before-snapshot:', c3.SNAP.relative_to(ROOT) / 'before_federal_aum.json')
print('taken', snap['snapshot'], snap['taken'], '| baseline', snap['baseline_schema_version'], '| engine', snap['engine_schema_version'])
print('engine forage_aum.federal, statewide:', round(sum(c['engine_forage_aum']['federal'] for c in snap['counties'].values())))
print('before blm_authorized_aum non-null counties:', sum(c['federal_aums']['blm_authorized_aum']['value'] is not None for c in snap['counties'].values()))

before-snapshot: build_log/wave7/snapshots/w7-f3/before_federal_aum.json
taken before_c3 2026-11-16 | baseline wy_county_ag_baseline_v1 | engine county-ag-engine-v1
engine forage_aum.federal, statewide: 201955
before blm_authorized_aum non-null counties: 0


In [3]:
# C-3 cell 3 — ingest, join RAS <-> polygons on ST_ALLOT, report match rate both directions
res = c3.build()
print('RAS columns dropped at ingest (not in allowlist):', res['dropped_cols'])
print('SHA-256 RAS:', c3.sha256(c3.RAS_CSV))
stats = c3.join_stats(res)
print(json.dumps(stats, indent=1, default=float))
j = res['joined']
print()
print('UNMATCHED RAS allotments (have AUMs, no polygon) -> not apportioned:')
print(j[j['join'] == 'ras_only'][['st_allot', 'allot_name', 'admin_st', 'public_acres', 'active_aum', 'suspended_aum']].to_string(index=False))
print()
print('UNMATCHED polygons (no RAS row): n=%d, acres=%.0f; 10 largest:' % (stats['polygon_only_count'], stats['polygon_only_acres']))
print(j[j['join'] == 'polygon_only'].nlargest(10, 'poly_acres')[['st_allot', 'poly_name', 'poly_acres']].to_string(index=False))
print()
print('Matched allotments with NO Wyoming overlap (out-of-state, excluded):')
print(j[(j['join'] == 'matched') & (j['wy_share'] == 0)][['st_allot', 'allot_name', 'active_aum']].to_string(index=False))
print()
print('Matched allotments crossing the state line (Wyoming share only):')
print(j[(j['join'] == 'matched') & (j['wy_share'] > 0) & (j['wy_share'] < 0.99)][['st_allot', 'allot_name', 'active_aum', 'wy_share']].round(3).to_string(index=False))
print()
print('Multi-polygon allotments:', j[(j['join'] == 'matched') & (j['polygons'] > 1)]['st_allot'].tolist(), '(dissolved to one geometry)')
print('Allotments with >1 RAS row:', stats['allotments_multi_ras_rows'], '(max %d; AUMs summed per allotment)' % stats['max_ras_rows_one_allotment'])

RAS columns dropped at ingest (not in allowlist): ['AMP Implement Date', 'Amp Text', 'Authorization Number', 'Field Office', 'Field Office.1', 'Grazing Decision']
SHA-256 RAS: 97faa8cf55f7285136ebb82827e7aeeeba5c8e2720230f22d2fc7d07b5a89482
{
 "ras_allotments": 3503,
 "polygon_allotments": 3556,
 "matched": 3470,
 "match_rate_ras_to_polygon": 0.9905795032829003,
 "match_rate_polygon_to_ras": 0.9758155230596175,
 "ras_rows": 4128,
 "ras_active_aum_total_all_rows": 1923797.0,
 "ras_active_aum_wy_admin": 1923355.0,
 "ras_suspended_aum_wy_admin": 393770.0,
 "ras_only_count": 33,
 "ras_only_active_aum": 8603.0,
 "ras_only_public_acres": 152244.0,
 "polygon_only_count": 86,
 "polygon_only_acres": 208099.5915359915,
 "matched_active_aum": 1915194.0,
 "matched_zero_wy_overlap_count": 20,
 "matched_zero_wy_overlap_aum": 851.0,
 "matched_state_line_crossing_count": 12,
 "apportioned_active_aum_wy": 1908244.8710056767,
 "apportioned_suspended_aum_wy": 390189.85084935185,
 "allotments_multi_polygo

In [4]:
# C-3 cell 4 — county apportionment and AMENDED credibility gate (W7-F5; printed, not annotated)
# Replaces the F3 named-county gate (DECISIONS.md 2026-11-16 b). Tolerances were set after seeing the data.
baseline = json.loads((c3.PROCESSED / 'wy_county_ag_baseline.json').read_text())
cf = c3.county_frame(res, baseline)
print(cf[['county_name', 'blm_active_aum', 'blm_suspended_aum', 'blm_public_acres', 'blm_acres_share_of_county',
          'sma_blm_acres', 'ras_over_sma_blm_acres', 'blm_share_of_forage_capacity']].round(3).to_string(index=False))
print()
print('Published source:', c3.PUBLISHED_SRC)
gate = pd.DataFrame(c3.gate_rows(stats, cf))
print(gate[['result', 'check', 'value', 'expected', 'deviation']].to_string(index=False))
cg = c3.county_acre_gate(cf)
print('\nPer-county acre gate: %d/%d PASS; max |deviation| = %.2f%% (%s), max |diff| = %.0f acres'
      % ((cg.result == 'PASS').sum(), len(cg), cg.dev_pct.abs().max(), cg.loc[cg.dev_pct.abs().idxmax(), 'county_name'], cg.diff_acres.abs().max()))
print('Failures:', cg[cg.result == 'FAIL'][['county_name', 'diff_acres', 'dev_pct', 'tolerance_acres', 'exception']].round(1).to_dict('records'))
print('Pass only via the 5,000-acre floor:', sorted(cg[(cg.result == 'PASS') & cg.via_floor].county_name))
print('GATE PASSES (no exceptions applied):', bool((gate.result == 'PASS').all()))
print('\nSanity note (NOT gating; F3 named-county list, written by the PM from memory):')
print(pd.DataFrame(c3.sanity_notes(cf)).to_string(index=False))
print('Sensitivity (JC-3 literal reading, AUM x area share x public proportion): %.0f AUMs statewide, %.1f%% of published'
      % (cf.blm_aum_literal_pp.sum(), 100 * cf.blm_aum_literal_pp.sum() / c3.PUBLISHED_WY_ACTIVE_AUM))

county_name  blm_active_aum  blm_suspended_aum  blm_public_acres  blm_acres_share_of_county  sma_blm_acres  ras_over_sma_blm_acres  blm_share_of_forage_capacity
     Albany       48336.371             35.000        290029.787                      0.106     294933.063                   0.983                         0.545
   Big Horn       71250.777          56956.107       1138731.461                      0.567    1162300.253                   0.980                         0.866
   Campbell       38171.257            124.000        225434.948                      0.073     224650.695                   1.003                         0.320
     Carbon      300330.286           1748.682       2022494.462                      0.400    2047543.290                   0.988                         0.773
   Converse       24967.053              0.000        131460.892                      0.048     129767.548                   1.013                         0.245
      Crook       14943.853       

In [4]:
# C-3 cell 4b — DIAGNOSTIC ONLY (W7-F5). Does not alter the gate or any output.
# BLM surface acres in the independent ownership layer that intersect no allotment polygon (EPSG:5070).
d = c3.nonallotted_diagnostic(cf)
d['non_allotted_as_share_of_shortfall'] = (d.sma_blm_not_in_any_allotment_acres / d.gap_acres_negative.where(d.gap_acres_negative > 0)).round(2)
print(d[['county_name', 'diff_acres', 'dev_pct', 'tolerance_acres', 'result', 'sma_blm_not_in_any_allotment_acres',
         'non_allotted_as_share_of_shortfall']].round(1).to_string(index=False))
print('Statewide non-allotted BLM acres: %.0f of %.0f ownership-layer BLM acres' % (d.sma_blm_not_in_any_allotment_acres.sum(), d.sma_blm_acres.sum()))

county_name  diff_acres  dev_pct  tolerance_acres result  sma_blm_not_in_any_allotment_acres  non_allotted_as_share_of_shortfall
     Albany     -4903.3     -1.7          14746.7   PASS                              4880.3                                 1.0
   Big Horn    -23568.8     -2.0          58115.0   PASS                             18800.1                                 0.8
   Campbell       784.3      0.3          11232.5   PASS                              2474.7                                 NaN
     Carbon    -25048.8     -1.2         102377.2   PASS                             18978.6                                 0.8
   Converse      1693.3      1.3           6488.4   PASS                              2478.1                                 NaN
      Crook      -425.7     -0.5           5000.0   PASS                              2866.8                                 6.7
    Fremont    -64487.9     -3.1         105095.2   PASS                             37210.2     

In [5]:
# C-3 cell 5 — write outputs (idempotent). Baseline v2, sidecars, manifest. NOT promoted to the engine.
c3.SNAP.mkdir(parents=True, exist_ok=True)
rep = c3.write_report(res, baseline)
c3.apply_baseline(res, rep)
c3.write_sidecars(rep['stats'])
c3.update_manifest(rep['stats'])
print('baseline schema:', json.loads((c3.PROCESSED / 'wy_county_ag_baseline.json').read_text())['schema_version'])
print('promoted to wy_county_ag_engine_baseline.json / terra-app: NO (see HANDOFF W7-F3)')

baseline schema: wy_county_ag_baseline_v2
promoted to wy_county_ag_engine_baseline.json / terra-app: NO (see HANDOFF W7-F3)


### Engine impact (scratch run, no repo fixture touched)

Candidate promotion = engine `forage_aum.federal` becomes BLM active AUMs + the existing USFS-acres proxy.
Statewide federal forage would go from 201,955 to 2,110,200 AUMs (x10.4).

| Fixture | Digests that move |
|---|---|
| Golden N (Python, `tests/test_golden_n.py` run unchanged against a scratch data dir) | **12 of 60**: `ag_digest_md5` in all 6 arcs x 2 lenses. State, fiscal, existing-assets, history digests unchanged. |
| Ranch Country 2040 (TS, scratch copy of `terra-app`) | `ag_digest_md5` 0c537942... -> 8e25daa2... (drought on); 756ec0d2... -> ee8c04e4... (energy-only) |

Promotion and golden regeneration are left for a dated decision. Full tables in HANDOFF W7-F3.